# Notebook 1: Extraction Pipeline Walkthrough

**Cookbook Example 1 — Real end-to-end extraction for nusinersen**

This notebook demonstrates the full pipeline:
1. Fetch the nusinersen SPL label from OpenFDA
2. Extract a structured triple with Claude
3. Cross-validate against ChEMBL
4. Show how the record lands in the validated JSONL

> **Prerequisites:**
> - `pip install -e '.[dev]'` from the repo root
> - `export ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
import json
import os
from pathlib import Path
from rich.console import Console
from rich.pretty import pprint

console = Console()

# Verify keys are set
assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY first'
print('Keys OK')

## Step 1: Fetch from OpenFDA

We query `https://api.fda.gov/drug/label.json` for nusinersen.
Results are cached to `data/raw/openfda/` so re-runs don't hit the API.

In [ ]:
from fda_strategy_triples.fetch.openfda import fetch_label_by_name, extract_label_text, get_openfda_metadata

label = fetch_label_by_name('nusinersen', use_cache=True)

if label:
    meta = get_openfda_metadata(label)
    print('=== OpenFDA metadata ===')
    pprint(meta)
else:
    print('Label not found — check your internet connection and API rate limits')

In [ ]:
# Show the mechanism-of-action section of the label
if label:
    moa = label.get('mechanism_of_action', ['(not in label)'])[0]
    print('=== Mechanism of Action (raw label text) ===')
    print(moa[:2000])

## Step 2: Augment with DailyMed

In [ ]:
from fda_strategy_triples.fetch.dailymed import fetch_label_by_name as dm_fetch, extract_sections

dm_label = dm_fetch('nusinersen', use_cache=True)
if dm_label:
    sections = extract_sections(dm_label)
    print('DailyMed sections found:', list(sections.keys()))
    if 'mechanism_of_action' in sections:
        print('\n=== DailyMed MoA snippet ===')
        print(sections['mechanism_of_action'][:800])
else:
    print('DailyMed label not found')

## Step 3: Extract with Claude

The extractor calls Claude with a structured tool-use prompt.
The tool schema matches the Pydantic `FDATriple` model exactly.
Prompt version is pinned so provenance is reproducible.

In [ ]:
from fda_strategy_triples.extract.extractor import extract_triple, PROMPT_VERSION, DEFAULT_MODEL
from fda_strategy_triples.extract.schema import DrugSeed, MechanismClass
from fda_strategy_triples.fetch.openfda import extract_label_text

seed = DrugSeed(
    drug_name_generic='nusinersen',
    drug_name_brand='Spinraza',
    gene_hints=['SMN1', 'SMN2'],
    mechanism_hint=MechanismClass.aso,
)

label_text = extract_label_text(label) if label else ''

print(f'Prompt version: {PROMPT_VERSION}')
print(f'Model: {DEFAULT_MODEL}')
print(f'Label text length: {len(label_text):,} chars')

In [ ]:
# This cell makes a real Claude API call (~$0.001)
record = extract_triple(
    seed,
    label_text,
    model=DEFAULT_MODEL,
    source_apis=['openfda'],
)

print('=== Extracted Triple ===')
print(record.model_dump_json(indent=2))

## Step 4: Cross-validate against ChEMBL

ChEMBL ID for nusinersen: `CHEMBL1950026`

In [ ]:
from fda_strategy_triples.fetch.chembl import extract_validation_fields

chembl_data = extract_validation_fields('CHEMBL1950026')
print('=== ChEMBL validation data ===')
pprint(chembl_data)

In [ ]:
from fda_strategy_triples.validate.cross_check import cross_check_record

updated_record, discrepancies = cross_check_record(
    record,
    chembl_id='CHEMBL1950026',
)

print('ChEMBL validated:', updated_record.metadata.chembl_validated)
print('Discrepancies:', discrepancies or 'none')

## Step 5: Compare with the human-reviewed ground truth

The validated record for nusinersen is already in `data/validated/validated_triples.jsonl`.
Let's compare the key fields.

In [ ]:
validated_path = Path('../data/validated/validated_triples.jsonl')
validated_records = [
    json.loads(line)
    for line in validated_path.read_text().splitlines()
    if line.strip()
]

nusinersen_gt = next(
    (r for r in validated_records if r['triple']['drug_name_generic'] == 'nusinersen'),
    None
)

if nusinersen_gt and record:
    print('=== Field comparison: Extracted vs Ground Truth ===')
    fields_to_compare = ['variant_context', 'molecular_target', 'mechanism_class', 'mechanism_summary']
    for field in fields_to_compare:
        extracted_val = getattr(record.triple, field)
        gt_val = nusinersen_gt['triple'][field]
        match = '✓' if str(extracted_val) == str(gt_val) else '≈'
        print(f'\n[{match}] {field}')
        print(f'  EXTRACTED: {extracted_val}')
        print(f'  GT:        {gt_val}')

## Cookbook Example 1 Summary

| Step | Tool | Output |
|---|---|---|
| Fetch | OpenFDA API | Raw SPL JSON (cached) |
| Augment | DailyMed API | Additional sections |
| Extract | Claude `claude-sonnet-4-6`, prompt `v1.2.0` | `ExtractedTriple` |
| Validate | ChEMBL `CHEMBL1950026` | Discrepancy list |
| Review | Human (interactive `fda-review`) | `validated=True` |

Cost per drug: ~\$0.001–0.005 in Claude API tokens (varies by label length).